# 批量归一化层

当神经网络比较深的时候，forward函数从下往上，backward函数从上往下，梯度一般是比较小的值，会有梯度消失的风险

而且，数据是在最底部的：

底部的层的训练比较慢

底部层一旦发生变化，所有的数据都会发生变化

最后底部的层需要重新学习很多很多次，收敛会变得很慢

批量归一化的目的是：在改变底部层的参数的情况下，可以避免顶部层的参数变化

核心想法是：（既然数据分布老是变，那我就在每一层强行把它们拉回到一个标准的分布）

*在分布固定住的情况下，固定小批量里面的均值和方差，然后再去做额外的调整*

具体来说，在模型训练时，对于一个小批量（Batch）的数据：
求均值和方差： 计算这个 Batch 内数据的均值 $\mu$ 和方差 $\sigma^2$:

$U_{B} = \frac{1}{|B|} \sum_{i \in B} x_i and \delta^2 = \frac{1}{|B|} \sum_{i \in B} (x_i - U_{B})^2 + \epsilon$


标准化： 把数据减去均值，除以标准差，使其变成均值为 0，方差为 1 的分布。$$x_i \leftarrow \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$(其中 $\epsilon$ 是一个很小的数，防止分母为 0)

*缩放和平移*

如果只是生硬地把数据拉回正态分布，可能会破坏网络本来已经学到的特征表示。因此，BN 引入了两个可学习的参数：拉伸参数 $\gamma$（Gamma）和偏移参数 $\beta$（Beta）。去做额外的调整$$x_{i+1} = \gamma \frac{x_i - u_B}{\delta_B} + \beta$$

网络在训练时会自己学习这两个参数。如果网络认为这层的数据不需要标准化，它完全可以学出 $\gamma = \sigma$, $\beta = \mu$ 来抵消掉标准化的操作。

BN层作用与：

* 全连接层和卷积层的输出之上，激活函数之前
* 全连接层和卷积层的输入上

对于全连接层，作用在特征维（对于每一个特征去计算标量的均值和方差，使得每一个特征变成均值为0方差为1）

对于卷积层，作用在通道维

解释一下卷积层：

对于一个批量大小\*高\*宽\*通道数的卷积层，样本数就是批量大小\*高\*宽,即整个批量里面所有的像素都是一个样本，通道数才是卷积层的特征

## 批量归一化到底是在干什么？

初始论文是想用这个方式减少内部的协变量转移

实际上这个方式就是在每个小批量里面加入噪音(加入的随机偏移和随机缩放)控制模型的复杂度：
$$
x_{i+1} = \gamma \frac{x_i - u_B}{\delta_B} + \beta
$$

没有必要和丢弃法去混合使用，dropout的时机是relu之后

## 代码实现

In [2]:
import torch
from torch import nn
from d2l import torch as d2l
# 核心思想是按照特征去求取均值和方差

def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum): # X表示二维输入，gamma和beta是超参数，这里的moving_mean和moving_var是全局的均值和方差，eps是用于除0的一个小常量， momentum是用来更新moving_mean和moving_var的参数
    # 通过is_grad_enabled来判断当前模式是训练模式还是预测模式

    if not torch.is_grad_enabled():  # 如果是在预测模式下，is_grad_enabled用来判断当前模式是不是训练模式，是训练模式返回True，是预测模式返回False
        # 如果是在预测模式下，直接使用传入的移动平均所得的均值和方差
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)  # 做预测模式的情况下就直接用全局的均值和方差
    else:
        assert len(X.shape) in (2, 4)   # 2表示全连接层，4表示卷积层
        if len(X.shape) == 2:
            # 使用全连接层的情况，计算特征维上的均值和方差
            mean = X.mean(dim=0)  # 样本维度上的均值
            var = ((X - mean) ** 2).mean(dim=0)  # 这里不加keepdim是因为有自动广播机制
        else:
            # 使用二维卷积层的情况，计算通道维上（axis=1）的均值和方差。（卷积层的通道维度是他的特征信息）
            # 这里我们需要保持X的形状以便后面可以做广播运算
            mean = X.mean(dim=(0, 2, 3), keepdim=True)  # 在axis=1的维度上进行计算，加了keepdim得到的数据为（1， C， 1， 1）
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
        # 训练模式下，用当前的均值和方差做标准化
        X_hat = (X - mean) / torch.sqrt(var + eps)
        # 更新移动平均的均值和方差
        moving_mean = momentum * moving_mean + (1.0 - momentum) * mean  # 在训练的时候会更新全局的均值和方差
        moving_var = momentum * moving_var + (1.0 - momentum) * var
    Y = gamma * X_hat + beta  # 缩放和移位
    return Y, moving_mean.data, moving_var.data

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 创建一个BatchNorm层

In [ ]:
class BatchNorm(nn.Module):
    # num_features：完全连接层的输出数量或卷积层的输出通道数。
    # num_dims：2表示完全连接层，4表示卷积层
    def __init__(self, num_features, num_dims):
        super().__init__()
        if num_dims == 2:
            shape = (1, num_features)  # num_features是特征维度
        else:
            shape = (1, num_features, 1, 1)
        # 参与求梯度和迭代的拉伸和偏移参数，分别初始化成1和0
        self.gamma = nn.Parameter(torch.ones(shape))  # gamma去拟合方差，交给框架去反向传播自动更新
        self.beta = nn.Parameter(torch.zeros(shape))    # beta去拟合均值
        # 非模型参数的变量初始化为0和1
        self.moving_mean = torch.zeros(shape)
        self.moving_var = torch.ones(shape)  # 这两个不需要迭代

    def forward(self, X):
        # 如果X不在内存上，将moving_mean和moving_var
        # 复制到X所在显存上
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)  # 迁移都GPU上进行计算
        # 保存更新过的moving_mean和moving_var
        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean,
            self.moving_var, eps=1e-5, momentum=0.9)
        return Y

## batchnorm层应用于LeNet模型

In [ ]:
net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5), BatchNorm(6, num_dims=4), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), BatchNorm(16, num_dims=4), nn.Sigmoid(),  # 第一层卷积之后加入BatchNorm层
    nn.AvgPool2d(kernel_size=2, stride=2), nn.Flatten(),
    nn.Linear(16*4*4, 120), BatchNorm(120, num_dims=2), nn.Sigmoid(),
    nn.Linear(120, 84), BatchNorm(84, num_dims=2), nn.Sigmoid(),
    nn.Linear(84, 10))
X = torch.randn((1, 1, 28, 28))  # 这里是针对LeNet模型的输入维度（28*28的输入）
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, X.shape)

Conv2d torch.Size([1, 6, 24, 24])
BatchNorm torch.Size([1, 6, 24, 24])
Sigmoid torch.Size([1, 6, 24, 24])
AvgPool2d torch.Size([1, 6, 12, 12])
Conv2d torch.Size([1, 16, 8, 8])
BatchNorm torch.Size([1, 16, 8, 8])
Sigmoid torch.Size([1, 16, 8, 8])
AvgPool2d torch.Size([1, 16, 4, 4])
Flatten torch.Size([1, 256])
Linear torch.Size([1, 120])
BatchNorm torch.Size([1, 120])
Sigmoid torch.Size([1, 120])
Linear torch.Size([1, 84])
BatchNorm torch.Size([1, 84])
Sigmoid torch.Size([1, 84])
Linear torch.Size([1, 10])


训练这里跑不动了，看一下每一层的结构吧